# Bank Marketing - Model Experiments


## 1. Introduction

이 노트북에서는 정기예금 가입 레코드의 순위를 산정할 모델 구성을 선택하기 위해
후보 모델을 비교하고 유망한 모델 계열의 hyperparameter를 tuning한다.

- Sentinel을 분리한 Logistic Regression baseline을 재현한다.
- `duration`을 제외하고 예측 시점에 사용할 수 있는 나머지 19개 예측 변수를 유지한다.
- 기존의 시간 순서에 따른 데이터 분할을 유지한다.
- Train 내부의 time-aware CV로 모델 계열을 비교하고 hyperparameter를 tuning한다.
- 각 training fold 안에서 전처리와 모델을 포함한 전체 Pipeline을 독립적으로 fit한다.
- Train에서 tuning한 후보를 Validation에서 baseline과 비교한다.
  AP를 주요 지표로 사용하고 ROC-AUC와 Top-5/10/20% 지표를 보완 근거로 활용한다.
- 성능과 함께 시간에 따른 안정성, 추론 비용, 모델 복잡도를 고려한다.
- 최종 평가 전에 feature 정책, 전처리, 모델, 파라미터, 평가 규칙을 확정한다.
- Test는 최종 평가를 위해 보류한다.

확정한 구성을 Train+Validation에 학습하고 독립적인 Test 평가를 수행하는 작업은
최종 평가 노트북에서 다룬다.


## 2. Data & Pipeline Setup


### 2.1 Load Data & Splits


원본 행 순서를 유지하고 기존의 시간 순서에 따른 경계를 사용해 Train 28,831행과
Validation 8,238행을 준비한다. Test를 로딩하지 않도록 앞의 37,069행만 읽는다.
이 경계는 전체 데이터 기준이며, 개발용 부분집합의 길이로 다시 계산하지 않는다.

`duration`과 target `y`를 제외한 예측 시점의 입력 변수 19개를 유지한다.
Target은 `no = 0`, `yes = 1`로 인코딩한다.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA_PATH = (
    PROJECT_ROOT / "data" / "raw" / "bank-additional" / "bank-additional-full.csv"
)

# Established boundaries from the full 41,188-row dataset.
train_end = 28831
valid_end = 37069

df = pd.read_csv(RAW_DATA_PATH, sep=";", nrows=valid_end)
train_df = df.iloc[:train_end].copy()
valid_df = df.iloc[train_end:valid_end].copy()

In [2]:
target = "y"
excluded_features = ["duration"]
modeling_features = [
    column for column in df.columns if column not in [target, *excluded_features]
]

X_train = train_df[modeling_features].copy()
X_valid = valid_df[modeling_features].copy()

target_mapping = {"no": 0, "yes": 1}
y_train = train_df[target].map(target_mapping)
y_valid = valid_df[target].map(target_mapping)

pd.DataFrame(
    {
        "rows": [len(X_train), len(X_valid)],
        "features": [X_train.shape[1], X_valid.shape[1]],
        "positive_rate": [y_train.mean(), y_valid.mean()],
    },
    index=["Train", "Validation"],
)

,rows,features,positive_rate
Train,28831,19,0.055704
Validation,8238,19,0.138626


분할 요약은 이후 지표 비교의 맥락을 제공한다. Train과 Validation은 서로 다른 기간을
포함하며 양성 비율도 다르므로 AP와 Precision@K를 해당 비율과 함께 해석해야 한다.


### 2.2 Sentinel-Aware Preprocessing


전처리 및 baseline 노트북과 같은 표현을 사용한다.

| Feature 그룹 | 변환 |
|---|---|
| 일반 수치형 feature 8개 | `StandardScaler` |
| `pdays` | 이전 연락 여부와 경과 일수로 분리한 뒤 `StandardScaler` |
| 범주형 feature 10개 | `OneHotEncoder(handle_unknown="ignore")` |

`pdays = 999`인 행은 indicator와 경과 일수를 모두 `0`으로 바꾼다.
나머지 행은 indicator를 `1`로 설정하고 기록된 경과 일수를 유지한다.
Sentinel과 실제 0일 간격을 구분하려면 두 열이 모두 필요하다.
문자열 `"unknown"`은 범주로 유지한다.


In [3]:
numeric_features = [
    "age",
    "campaign",
    "previous",
    "emp.var.rate",
    "cons.price.idx",
    "cons.conf.idx",
    "euribor3m",
    "nr.employed",
]

categorical_features = [
    "job",
    "marital",
    "education",
    "default",
    "housing",
    "loan",
    "contact",
    "month",
    "day_of_week",
    "poutcome",
]


def split_pdays(values):
    pdays = np.asarray(values).reshape(-1)
    previously_contacted = (pdays != 999).astype(float)
    elapsed_days = np.where(pdays == 999, 0.0, pdays.astype(float))
    return np.column_stack([previously_contacted, elapsed_days])


def pdays_feature_names(transformer, input_features):
    return np.asarray(
        ["pdays_previously_contacted", "pdays_elapsed"],
        dtype=object,
    )

In [4]:
pdays_transformer = Pipeline(
    steps=[
        (
            "derive",
            FunctionTransformer(
                split_pdays,
                validate=False,
                feature_names_out=pdays_feature_names,
            ),
        ),
        ("scale", StandardScaler()),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features,
        ),
        ("pdays", pdays_transformer, ["pdays"]),
    ],
    remainder="drop",
)

baseline_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(max_iter=1000)),
    ]
)

전체 Pipeline은 전처리와 모델을 함께 묶는다. 구성하는 시점에는 학습하지 않는다.
Baseline은 Train에서만 fit하며, 이후 CV에서는 각 training fold 안에서
새로운 전체 Pipeline을 fit해야 한다.


## 3. Baseline Reproduction


### 3.1 Training & Validation Evaluation


Baseline 전처리와 `LogisticRegression(max_iter=1000)`을 Train에서 fit하고
Validation의 반올림하지 않은 양성 클래스 점수를 계산한다.

AP를 주요 지표로 사용하고 ROC-AUC로 판별력을 보완해 평가한다.
Top-5/10/20%에서는 `K = ceil(n_records * fraction)`을 사용한다.
점수는 내림차순 stable sort로 정렬해 동점일 때 원본 순서를 유지한다.
Lift의 기준은 Validation 양성 비율이다.


In [5]:
baseline_pipeline.fit(X_train, y_train)
y_valid_score = baseline_pipeline.predict_proba(X_valid)[:, 1]

fitted_model = baseline_pipeline.named_steps["model"]
pd.Series(
    {
        "optimization_iterations": fitted_model.n_iter_[0],
        "transformed_features": fitted_model.n_features_in_,
        "validation_scores": len(y_valid_score),
    },
    name="value",
)

optimization_iterations      35
transformed_features         62
validation_scores          8238
Name: value, dtype: int64

In [6]:
validation_prevalence = y_valid.mean()
baseline_ap = average_precision_score(y_valid, y_valid_score)
baseline_roc_auc = roc_auc_score(y_valid, y_valid_score)

general_results = pd.Series(
    {
        "validation_prevalence": validation_prevalence,
        "average_precision": baseline_ap,
        "roc_auc": baseline_roc_auc,
    },
    name="Logistic Regression",
)

general_results.to_frame().round(6)

,Logistic Regression
validation_prevalence,0.138626
average_precision,0.288234
roc_auc,0.688023


In [7]:
ranked_indices = np.argsort(-y_valid_score, kind="stable")
ranked_y = y_valid.to_numpy()[ranked_indices]
cumulative_positives = np.cumsum(ranked_y)
total_positives = int(y_valid.sum())

ranking_rows = []
for fraction in [0.05, 0.10, 0.20]:
    k = int(np.ceil(len(y_valid) * fraction))
    captured = int(cumulative_positives[k - 1])
    precision = captured / k

    ranking_rows.append(
        {
            "top_fraction": fraction,
            "k": k,
            "captured_subscribers": captured,
            "precision_at_k": precision,
            "recall_at_k": captured / total_positives,
            "lift_at_k": precision / validation_prevalence,
        }
    )

ranking_results = pd.DataFrame(ranking_rows)
ranking_results.round(6)

,top_fraction,k,captured_subscribers,precision_at_k,recall_at_k,lift_at_k
0,0.05,412,179,0.434466,0.156743,3.134090
1,0.10,824,309,0.375000,0.270578,2.705123
2,0.20,1648,499,0.302791,0.436953,2.184233


Top-K 지표는 레코드 단위의 용량별 순위 성능이며, 고유 고객 수나 일일 연락 할당량을
재현하지 않는다. 점수는 검증된 가입 확률이 아니며, 예측 지표는 수익성이나
연락의 인과효과를 입증하지 않는다.


### 3.2 Comparison with Stored Baseline Results


아래 기준값은 [Baseline 모델링](03-baseline-modeling.ipynb)에 저장된 Validation 결과를
소수점 여섯 자리로 반올림한 값이다. 이번 실행에서 새로 계산한 값과 구분한다.

앞의 셀을 실행해 현재 결과와 기준값의 차이를 계산한다. 작은 차이는 기준값 반올림에서
발생할 수 있다. 차이가 크다면 후보 모델의 개선을 해석하기 전에 원인을 확인해야 한다.


In [8]:
stored_baseline = pd.Series(
    {
        "average_precision": 0.288234,
        "roc_auc": 0.688023,
        "precision_at_10pct": 0.375000,
        "recall_at_10pct": 0.270578,
        "lift_at_10pct": 2.705123,
    },
    name="stored_reference",
)

top_10 = ranking_results.loc[ranking_results["top_fraction"] == 0.10].iloc[0]
current_baseline = pd.Series(
    {
        "average_precision": baseline_ap,
        "roc_auc": baseline_roc_auc,
        "precision_at_10pct": top_10["precision_at_k"],
        "recall_at_10pct": top_10["recall_at_k"],
        "lift_at_10pct": top_10["lift_at_k"],
    },
    name="current_run",
)

baseline_comparison = pd.concat([stored_baseline, current_baseline], axis=1)
baseline_comparison["difference"] = current_baseline - stored_baseline
baseline_comparison

,stored_reference,current_run,difference
average_precision,0.288234,0.288234,3.437315e-07
roc_auc,0.688023,0.688023,-2.462303e-08
precision_at_10pct,0.375000,0.375000,0.000000e+00
recall_at_10pct,0.270578,0.270578,-6.654991e-08
lift_at_10pct,2.705123,2.705123,-4.080560e-07


2026-10-26에 새 커널에서 당시 작성된 코드 셀 8개를 위에서 아래로 실행해
저장된 baseline 지표를 소수점 여섯 자리까지 재현했다. AP는 `0.288234`,
ROC-AUC는 `0.688023`, Lift@10%는 `2.705123`이다.
비교한 5개 지표의 차이는 모두 `5e-7` 미만으로, 저장된 기준값의 반올림 범위와 일치한다.

이 baseline은 후보 비교를 위한 재현 가능한 기준이며, 새로운 성능 개선이나
최종 모델 선택을 의미하지 않는다. 아래 모델 계열 비교 코드는 아직 실행하지 않았고
hyperparameter tuning은 구현하지 않았다. 당시 실행에서는 Train과 Validation만
로딩했으며 Test는 제외했다.

이전 원본 `pdays` 결과는 과거 근거로 남긴다. 현재 sentinel 처리 정책에서는
선택 가능한 표현이나 현재 baseline으로 사용하지 않는다.


## 4. Model-Family Comparison


### 4.1 Hypotheses & Candidate Models


동일한 feature 정책과 sentinel을 분리한 전처리 아래에서 세 후보를 비교한다.
Logistic Regression은 재현한 baseline과 동일한 설정을 유지한다.

| 후보 | 가설 |
|---|---|
| Logistic Regression | 시간에 따른 분포 변화 속에서도 선형 baseline이 경쟁력을 유지할 수 있다. |
| Random Forest | 비선형 관계와 feature 간 상호작용을 학습하면 순위 성능이 개선될 수 있다. |
| RBF SVM | Kernel 기반 비선형 경계가 선형 baseline보다 순위 성능을 개선할 수 있다. |

이 단계에서는 파라미터 탐색 대신 초기 구성을 비교한다. Random Forest는 tree 100개,
RBF SVM은 `C=1.0`, `gamma="scale"`을 사용한다. 나머지 모델 파라미터는 설치된 버전의
기본값을 유지하고, Random Forest의 `random_state=42`를 지정한다.
다른 실험 요인을 추가하지 않도록 class weight도 변경하지 않는다.
비교 결과는 특정 초기 설정에 대한 것이며 각 모델 계열의 최고 성능을 의미하지 않는다.

Logistic Regression과 Random Forest는 반올림하지 않은 양성 클래스 점수,
SVM은 decision-function 점수를 사용한다. AP와 ROC-AUC는 두 점수 유형을 모두 지원하므로
SVM의 확률 추정이나 calibration은 추가하지 않는다.


In [ ]:
from time import perf_counter

from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.svm import SVC

candidate_models = [
    ("Logistic Regression", LogisticRegression(max_iter=1000)),
    (
        "Random Forest",
        RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=1),
    ),
    ("RBF SVM", SVC(kernel="rbf", C=1.0, gamma="scale")),
]

### 4.2 Time-Aware Cross-Validation


Train 내부에서 학습 구간이 확장되는 `TimeSeriesSplit` fold 4개를 사용한다.
각 학습 구간은 평가 구간보다 앞선 행만 포함하며 셔플 없이 원본 순서를 유지한다.
모든 후보에 같은 fold 경계를 적용한다.

구간은 레코드 수를 기준으로 정한다. 완전한 시각 정보가 없어 각 구간을 동일한 달력상 기간으로
해석하거나 연락 이력에 근거한 간격을 정하기 어렵다. `gap=0`을 사용하되,
이것이 반복 고객 간 의존성을 해결한다고 주장하지 않는다.
성능을 해석하기 전에 fold 크기와 양성 비율을 확인한다.

매번 원본 입력과 복제한 전체 Pipeline을 사용해 fit한다. Baseline 전처리기가 이미
전체 Train에서 학습됐더라도 복제본은 해당 fold의 학습 행에서만 scaler 통계와
encoder 범주를 학습한다. 이 비교에는 Validation과 Test를 사용하지 않는다.


In [ ]:
time_cv = TimeSeriesSplit(n_splits=4)
cv_splits = list(time_cv.split(X_train))

fold_rows = []
for fold, (train_indices, eval_indices) in enumerate(cv_splits, start=1):
    fold_rows.append(
        {
            "fold": fold,
            "train_start": int(train_indices[0]),
            "train_end_exclusive": int(train_indices[-1] + 1),
            "eval_start": int(eval_indices[0]),
            "eval_end_exclusive": int(eval_indices[-1] + 1),
            "train_rows": len(train_indices),
            "eval_rows": len(eval_indices),
            "train_prevalence": y_train.iloc[train_indices].mean(),
            "eval_prevalence": y_train.iloc[eval_indices].mean(),
        }
    )

fold_summary = pd.DataFrame(fold_rows)
fold_summary

후보를 순차적으로 학습하고 fold별 AP와 ROC-AUC를 기록한다.
실제 경과 시간 기준으로 학습과 점수 계산 시간을 따로 측정한다.
두 시간에는 전처리가 포함되며 실행 환경의 영향을 받는다.
병렬 실행이 비용 비교에 영향을 주지 않도록 Random Forest는 worker 하나를 사용한다.
RBF SVM은 학습 구간이 커질수록 실행 시간이 크게 늘어날 수 있다.


In [ ]:
cv_rows = []
for model_name, estimator in candidate_models:
    for fold, (train_indices, eval_indices) in enumerate(cv_splits, start=1):
        X_fold_train = X_train.iloc[train_indices]
        y_fold_train = y_train.iloc[train_indices]
        X_fold_eval = X_train.iloc[eval_indices]
        y_fold_eval = y_train.iloc[eval_indices]

        fold_pipeline = Pipeline(
            steps=[
                ("preprocessor", clone(preprocessor)),
                ("model", clone(estimator)),
            ]
        )

        fit_start = perf_counter()
        fold_pipeline.fit(X_fold_train, y_fold_train)
        fit_seconds = perf_counter() - fit_start

        score_start = perf_counter()
        if model_name == "RBF SVM":
            fold_scores = fold_pipeline.decision_function(X_fold_eval)
        else:
            fold_scores = fold_pipeline.predict_proba(X_fold_eval)[:, 1]
        score_seconds = perf_counter() - score_start

        cv_rows.append(
            {
                "model": model_name,
                "fold": fold,
                "average_precision": average_precision_score(y_fold_eval, fold_scores),
                "roc_auc": roc_auc_score(y_fold_eval, fold_scores),
                "fit_seconds": fit_seconds,
                "score_seconds": score_seconds,
            }
        )
        print(f"{model_name}, fold {fold}: completed")

cv_results = pd.DataFrame(cv_rows)
cv_results.merge(
    fold_summary[["fold", "eval_rows", "eval_prevalence"]],
    on="fold",
    how="left",
).round(6)

### 4.3 Results & Tuning Candidates


**평가.** Fold별 AP의 단순 평균으로 초기 설정의 순위를 정한다.
양성 비율, 보조 지표인 ROC-AUC, 계산 비용과 함께 각 fold의 AP를 살펴본다.
표준편차는 기간에 따른 변동을 나타낸다. 확장되는 학습 구간이 겹치고 fold가 독립적이지
않으므로 이를 신뢰구간으로 해석하지 않는다.

후보는 동일한 fold 안에서 비교한다. 평균 CV AP와 이후 Validation AP의 차이는
서로 다른 기간과 양성 비율의 영향을 받으므로 모델 개선으로 해석하지 않는다.
Top-5/10/20% 지표는 이후 Validation 비교에서 평가한다.


In [ ]:
cv_summary = (
    cv_results.groupby("model", sort=False)
    .agg(
        mean_ap=("average_precision", "mean"),
        std_ap=("average_precision", "std"),
        min_ap=("average_precision", "min"),
        mean_roc_auc=("roc_auc", "mean"),
        total_fit_seconds=("fit_seconds", "sum"),
        total_score_seconds=("score_seconds", "sum"),
    )
    .sort_values("mean_ap", ascending=False, kind="stable")
)
cv_summary.round(6)

In [ ]:
fold_ap_comparison = cv_results.pivot(
    index="fold", columns="model", values="average_precision"
)
fold_ap_comparison["eval_prevalence"] = fold_summary.set_index("fold")[
    "eval_prevalence"
]
fold_ap_comparison.round(6)

**결정 — 실행 대기.** 이 절은 아직 실행하지 않았으므로 우수한 모델 계열이나 tuning 후보를
선정하지 않았다. 실행 후 fold 결과가 어떤 가설을 지지하는지, 차이가 여러 기간에 걸쳐
유지되는지, 추가 비용이 정당한지 기록한다. 제한된 범위로 tuning할 유망한 계열을 하나 또는
소수 선택하고, 재현한 Logistic Regression baseline은 이후 Validation 비교에 유지한다.
작은 성능 차이가 복잡성 증가를 정당화하지 못하면 더 단순한 후보를 우선한다.

공식 학습 자료: [TimeSeriesSplit](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.TimeSeriesSplit.html),
[Random Forest](https://scikit-learn.org/stable/modules/ensemble.html#forest),
[Support Vector Machines](https://scikit-learn.org/stable/modules/svm.html).


## 5. Hyperparameter Tuning


### 5.1 Hypotheses & Search Scope


### 5.2 Train-Only CV Search


### 5.3 Results & Candidate Configurations


## 6. Validation Comparison


### 6.1 AP, ROC-AUC & Top-K Metrics


### 6.2 Results & Interpretation


### 6.3 Temporal Stability & Limitations


## 7. Model Selection & Final Configuration


## 8. Experiment Summary
